# Notebook 4 — Baseline Decision Tree (DEMO copy with steps)

**Research Project:** Lightweight Machine Learning-Based Cybersecurity Threat Detection for Resource-Constrained Hardware
**Author:** Anush Jindal

> **What this file is:** a worked demo of Stage 5. Type your own version first, in a new notebook called `04_baseline_model.ipynb`, using the steps below as your guide. Use this copy to compare against or when you are stuck.
>
> **What this file is not:** a results file. It has no outputs saved, and no number in it is real until *you* run it on *your* data.

## The goal of this stage

Train the simplest possible model (a Decision Tree with no tuning) on each of your four clue lists (A, B, C, D), and record how well it scores and how big and slow it is. This is the **baseline**: the reference point we will try to shrink later.

## Three rules

1. **Learn from `X_train` only.** `X_test` is the exam. Use it to score, never to choose clues or settings.
2. **Don't tune anything yet.** No `max_depth`, no pruning. That is Stage 7.
3. **Write down the numbers you actually get.** Never a guessed number.

## The steps

1. Setup
2. Load your data and your four clue lists
3. Test your own keep/remove claims
4. Train on clue list A and score it
5. Compare A, B, C, D
6. Shortcut check (does the tree just lean on port and window size?)
7. Save your results
8. Read your numbers honestly

## Step 1 — Setup

**What this does:** loads the tools and tells the notebook where your files live.

**Plain words:** `pandas` handles tables, `DecisionTreeClassifier` is the 20-questions model, `joblib` saves a model to a file so we can measure how big it is.

**Your turn:** change `DATA_DIR` if your Drive folder is named differently.

In [ ]:
import os, time, json, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/threat detector/data"
except ImportError:
    DATA_DIR = os.environ.get("THREAT_DATA_DIR", "data")

print("Using data folder:", DATA_DIR)

## Step 2 — Load your data and your four clue lists

**What this does:** reads the study pile (`X_train`, `y_train`), the exam pile (`X_test`, `y_test`), and the four lists of column names you saved in Notebook 3.

**Expected:** the shapes should match your notes from Notebook 2 (about 488,393 and 122,099 rows, 78 columns), and the four lists should have 57, 39, 15 and 6 names.

**If it breaks:** `FileNotFoundError` means a filename or folder is wrong. Open the Colab file browser (folder icon) and copy the exact name.

In [ ]:
X_train = pd.read_csv(os.path.join(DATA_DIR, "X_train_engineered.csv"))
X_test  = pd.read_csv(os.path.join(DATA_DIR, "X_test_engineered.csv"))
y_train = pd.read_csv(os.path.join(DATA_DIR, "y_train.csv")).squeeze("columns")
y_test  = pd.read_csv(os.path.join(DATA_DIR, "y_test.csv")).squeeze("columns")

with open(os.path.join(DATA_DIR, "feature_sets.json")) as f:
    feature_sets = json.load(f)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("y_train:", y_train.shape, " y_test:", y_test.shape)
print("Clue list sizes:", {name: len(cols) for name, cols in feature_sets.items()})
print()
print(y_train.value_counts())

assert list(X_train.columns) == list(X_test.columns), "train and test columns differ"
for name, cols in feature_sets.items():
    missing = [c for c in cols if c not in X_train.columns]
    assert not missing, f"Set {name} has names not in X_train: {missing}"

## Step 3 — Test your own keep/remove claims

**Why:** your feature table says some columns are *identical* to others or *always zero*. That came from reading about the dataset, not from your data. A scientist checks. If a claim turns out false, **the data wins**: fix your table and your `A_drop` list in Notebook 3.

**What to look for:**
- A pair printing `True` means the two columns really are identical, so dropping one is justified.
- A pair printing `False` means your reason was wrong. Decide again.
- In the table, `distinct_values = 1` means the column never changes, so it carries no information. A tiny `share_nonzero_%` means it is almost always zero.

In [ ]:
BENIGN_LABEL = "BENIGN"
assert BENIGN_LABEL in set(y_train), f"'{BENIGN_LABEL}' not found. Labels: {sorted(set(y_train))}"

claimed_identical = [
    ("Avg Fwd Segment Size", "Fwd Packet Length Mean"),
    ("Avg Bwd Segment Size", "Bwd Packet Length Mean"),
    ("Subflow Fwd Packets", "Total Fwd Packets"),
    ("Subflow Fwd Bytes", "Total Length of Fwd Packets"),
    ("Subflow Bwd Packets", "Total Backward Packets"),
    ("Subflow Bwd Bytes", "Total Length of Bwd Packets"),
    ("Fwd Header Length.1", "Fwd Header Length"),
    ("ECE Flag Count", "RST Flag Count"),
]
for a, b in claimed_identical:
    if a in X_train.columns and b in X_train.columns:
        print(f"{a!r} identical to {b!r}: {np.allclose(X_train[a], X_train[b])}")
    else:
        print(f"skipped (column not found): {a!r} / {b!r}")

print()
print("Spearman correlation (1.0 means same ordering, so a tree gains nothing from both):")
for a, b in [("Packet Length Variance", "Packet Length Std"), ("Average Packet Size", "Packet Length Mean")]:
    if a in X_train.columns and b in X_train.columns:
        print(f"  {a!r} vs {b!r}: {X_train[a].corr(X_train[b], method='spearman'):.4f}")

claimed_constant = [
    "Fwd PSH Flags", "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags", "CWE Flag Count",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",
]
rows = [
    {"column": c,
     "distinct_values": X_train[c].nunique(),
     "share_nonzero_%": round((X_train[c] != 0).mean() * 100, 4)}
    for c in claimed_constant if c in X_train.columns
]
pd.DataFrame(rows)

## Step 4 — Train on clue list A and score it

**What this does:**
1. `fit` — the tree studies `X_train` (with only the clues in the list) and the answer key `y_train`.
2. `predict` — the tree answers the exam questions in `X_test`.
3. We compare its answers with the real answers.

**The function below** does this for any clue list, and also measures how *expensive* the model is (time, size, number of questions), because that is our research question.

**Words to know:**
- **macro F1** — the average score across all classes, counting rare attacks as much as common ones. It is harsher than accuracy.
- **attack recall** — of all real attacks, how many did we catch? Missing one is the worst mistake.
- **false alarms** — normal traffic wrongly called an attack.
- **missed attacks** — real attacks wrongly called normal.
- **nodes / depth** — how many questions the tree contains, and how many it asks in a row. Smaller means cheaper.

**Why not just accuracy:** if 63% of traffic is normal, a lazy model that says "normal" always already scores 63%.

In [ ]:
def run_experiment(name, feats):
    model = DecisionTreeClassifier(random_state=42)

    t0 = time.perf_counter()
    model.fit(X_train[feats], y_train)
    train_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    pred = model.predict(X_test[feats])
    infer_s = time.perf_counter() - t0

    true_attack = (y_test != BENIGN_LABEL).to_numpy()
    pred_attack = pred != BENIGN_LABEL
    caught = int((true_attack & pred_attack).sum())
    missed = int((true_attack & ~pred_attack).sum())
    false_alarms = int((~true_attack & pred_attack).sum())

    p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average="macro", zero_division=0)

    path = os.path.join(DATA_DIR, f"tree_{name}.joblib")
    joblib.dump(model, path)

    result = {
        "experiment": name,
        "num_features": len(feats),
        "accuracy": accuracy_score(y_test, pred),
        "macro_precision": p,
        "macro_recall": r,
        "macro_f1": f1,
        "attack_recall": caught / max(caught + missed, 1),
        "false_alarms": false_alarms,
        "missed_attacks": missed,
        "nodes": int(model.tree_.node_count),
        "depth": int(model.get_depth()),
        "train_s": train_s,
        "infer_ms_per_1000_rows": infer_s / len(X_test) * 1_000_000,
        "model_size_kb": os.path.getsize(path) / 1024,
    }
    return result, model, pred


res_A, model_A, pred_A = run_experiment("A", feature_sets["A"])
results = [res_A]
pd.Series(res_A)

In [ ]:
print(classification_report(y_test, pred_A, zero_division=0, digits=4))

**The confusion matrix** is a score sheet. Each row is a *true* class, each column is what the tree *guessed*. A perfect tree has numbers only on the diagonal. Here each row is shown as a share of its class, so rare attacks are as visible as common ones.

**Your turn:** find a row where the diagonal is not near 1.0. Which class is it confused with? Write it in your notes.

In [ ]:
labels = sorted(set(y_test) | set(pred_A))
cm = confusion_matrix(y_test, pred_A, labels=labels)
cm_share = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_share, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.xlabel("What the tree guessed")
plt.ylabel("What it really was")
plt.title("Clue list A — share of each true class")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Step 5 — Compare A, B, C, D

**What this does:** repeats Step 4 for the smaller clue lists. This is the first piece of the research question: *how much do we lose when we use fewer clues?*

**Expected:** a table with four rows and two small graphs. **Do not decide anything from one run.** Just read it.

**Your turn:** before you look, guess whether the macro F1 will fall a lot or a little as clues shrink. Write your guess in your notes, then compare.

In [ ]:
for name in ["B", "C", "D"]:
    res, _, _ = run_experiment(name, feature_sets[name])
    results.append(res)

table = pd.DataFrame(results).sort_values("num_features", ascending=False).reset_index(drop=True)
table.round(4)

In [ ]:
main = table.sort_values("num_features")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(main["num_features"], main["macro_f1"], marker="o")
ax[0].set_xlabel("number of clues")
ax[0].set_ylabel("macro F1")
ax[0].set_title("Score vs number of clues")

ax[1].plot(main["num_features"], main["model_size_kb"], marker="o", color="tab:orange")
ax[1].set_xlabel("number of clues")
ax[1].set_ylabel("saved model size (KB)")
ax[1].set_title("Size vs number of clues")

plt.tight_layout()
plt.show()

## Step 6 — Shortcut check

**The worry (it is in your own feature table):** CICIDS2017 attacks were run from a lab against fixed ports and fixed machines. A tree might score well just by memorising *"this port or this window size means attack"* instead of learning how attacks behave. A real network would not hand us that shortcut.

**The test:** take those clues away and re-run. If the score barely moves, the tree is learning behaviour. If it collapses, it was leaning on the shortcut, and that is an important finding for the research.

**Careful:** this is *not* us choosing clues using the exam. We are asking a question about the model, not tuning it. Still, write down what you did and why.

In [ ]:
SHORTCUTS_PORT = ["Destination Port"]
SHORTCUTS_ALL = ["Destination Port", "Init_Win_bytes_forward", "Init_Win_bytes_backward"]

def without(cols, drop):
    return [c for c in cols if c not in drop]

shortcut_runs = [
    ("A_no_port", without(feature_sets["A"], SHORTCUTS_PORT)),
    ("A_no_port_no_window", without(feature_sets["A"], SHORTCUTS_ALL)),
    ("C_no_port", without(feature_sets["C"], SHORTCUTS_PORT)),
    ("D_no_port", without(feature_sets["D"], SHORTCUTS_PORT)),
]
for name, feats in shortcut_runs:
    res, _, _ = run_experiment(name, feats)
    results.append(res)

full_table = pd.DataFrame(results)
full_table.round(4)

## Step 7 — Save your results

**What this does:** writes your numbers to a CSV in the same column layout as `experiments/results.csv` in the repo.

**Then you:**
1. Download the CSV (or find it in your Drive folder).
2. Copy its rows into `experiments/results.csv` in the repo, and push.
3. Add a dated entry to `experiments/experiment_notes.md` saying what you saw and what surprised you.

`ram_mb` is left empty on purpose. We have not measured memory yet. That comes in the benchmarking stage, and we do not guess it.

In [ ]:
DATASET_FILE = "Wednesday-workingHours"
today = datetime.date.today().isoformat()

out = pd.DataFrame({
    "experiment_id": ["S5-" + n for n in full_table["experiment"]],
    "date": today,
    "stage": "5-baseline",
    "dataset_file": DATASET_FILE,
    "num_samples": len(X_train) + len(X_test),
    "num_features": full_table["num_features"],
    "features_used": full_table["experiment"],
    "max_depth": full_table["depth"],
    "num_nodes": full_table["nodes"],
    "accuracy": full_table["accuracy"].round(5),
    "precision": full_table["macro_precision"].round(5),
    "recall": full_table["macro_recall"].round(5),
    "f1": full_table["macro_f1"].round(5),
    "model_size_kb": full_table["model_size_kb"].round(1),
    "training_time_s": full_table["train_s"].round(2),
    "inference_ms": full_table["infer_ms_per_1000_rows"].round(3),
    "ram_mb": "",
    "notes": ("untuned DecisionTree(random_state=42); precision/recall/f1 are macro averages; "
              "inference_ms is per 1000 rows; attack_recall="
              + full_table["attack_recall"].round(5).astype(str)
              + " missed=" + full_table["missed_attacks"].astype(str)
              + " false_alarms=" + full_table["false_alarms"].astype(str)),
})

out_path = os.path.join(DATA_DIR, "results_stage5_baseline.csv")
out.to_csv(out_path, index=False)
print("Saved:", out_path)
out

## Step 8 — Read your numbers honestly

Before you write any conclusion, check each of these:

- **Did a very high score surprise you?** Don't celebrate yet. You split one day of traffic at random, so near-identical rows from the same attack burst can land in both piles. A high score here does not prove it would work on a different day or a different network.
- **Look at the rare attack classes** in the classification report. A class with only a handful of test rows (Heartbleed has very few) gives a score that can swing wildly. Report it, but don't build a claim on it.
- **Compare `attack_recall` with `accuracy`.** Missing real attacks (`missed_attacks`) matters more than a few false alarms.
- **Did the shortcut check change things?** If yes, say so plainly in your notes. That is a real finding, not a failure.
- **Never write a number you did not see on screen.**

## What comes next (do not start yet)

Stage 7 shrinks the tree on purpose (`max_depth`, pruning) and watches what happens to these same numbers. A good first question to bring to that stage: *at which point does the score start to fall faster than the size does?*